## Public portfolio method note
- Research question: Where does Icarus lose player trust across early and mid lifecycle stages?
- Input: Icarus Steam review exports from the local analysis folder.
- Filters: English-language reviews, cleaned text, playtime grouping, and survival/crafting/friction theme tagging.
- Output tables: `lifecycle_risk_map_pct.csv` and `complaint_portfolio_matrix.csv` inputs for the live-service dashboard.
- Limitations: Sentiment uses Steam recommendation labels as a proxy; raw review text is excluded from the public repo.


In [ ]:
import requests
import pandas as pd
import time

def fetch_steam_reviews(appid: str, game_name: str, max_reviews: int = 1500, language: str = "english") -> pd.DataFrame:
    """
    Fetch Steam reviews for a given appid.

    Parameters:
        appid (str): Steam app ID
        game_name (str): Game name
        max_reviews (int): Maximum number of reviews to fetch
        language (str): Review language, e.g. "english"

    Returns:
        pd.DataFrame: Reviews dataframe
    """
    url = f"https://store.steampowered.com/appreviews/{appid}"
    cursor = "*"
    rows = []

    while len(rows) < max_reviews:
        params = {
            "json": 1,
            "cursor": cursor,
            "num_per_page": 100,
            "language": language,
            "filter": "updated",
            "purchase_type": "all"
        }

        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Request failed: {e}")
            break

        reviews = data.get("reviews", [])
        if not reviews:
            print("No more reviews returned.")
            break

        for r in reviews:
            rows.append({
                "game": game_name,
                "platform": "Steam",
                "review_id": r.get("recommendationid"),
                "review_text": r.get("review"),
                "voted_up": r.get("voted_up"),
                "votes_up": r.get("votes_up"),
                "votes_funny": r.get("votes_funny"),
                "weighted_vote_score": r.get("weighted_vote_score"),
                "comment_count": r.get("comment_count"),
                "steam_purchase": r.get("steam_purchase"),
                "received_for_free": r.get("received_for_free"),
                "written_during_early_access": r.get("written_during_early_access"),
                "timestamp_created": r.get("timestamp_created"),
                "timestamp_updated": r.get("timestamp_updated"),
                "playtime_forever": r.get("author", {}).get("playtime_forever"),
                "playtime_last_two_weeks": r.get("author", {}).get("playtime_last_two_weeks"),
                "playtime_at_review": r.get("author", {}).get("playtime_at_review"),
                "last_played": r.get("author", {}).get("last_played"),
            })

            if len(rows) >= max_reviews:
                break

        cursor = data.get("cursor", cursor)
        print(f"Fetched {len(rows)} reviews so far...")
        time.sleep(1)

    df = pd.DataFrame(rows)

    if not df.empty:
        df["timestamp_created"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
        df["timestamp_updated"] = pd.to_datetime(df["timestamp_updated"], unit="s", errors="coerce")
        df["last_played"] = pd.to_datetime(df["last_played"], unit="s", errors="coerce")

        df["playtime_hours"] = df["playtime_forever"] / 60
        df["playtime_at_review_hours"] = df["playtime_at_review"] / 60
        df["review_length"] = df["review_text"].astype(str).str.len()

        df = df.drop_duplicates(subset=["review_id"])
        df = df.dropna(subset=["review_text"])
        df["review_text"] = df["review_text"].astype(str).str.strip()
        df = df[df["review_text"] != ""]

    return df


# Icarus appid
ICARUS_APPID = "1149460"

df_icarus = fetch_steam_reviews(
    appid=ICARUS_APPID,
    game_name="Icarus",
    max_reviews=1500,
    language="english"
)

print("\nFinished!")
print(df_icarus.shape)
print(df_icarus.head())

df_icarus.to_csv("icarus_steam_reviews_english_1500.csv", index=False, encoding="utf-8-sig")
print("Saved to icarus_steam_reviews_english_1500.csv")

In [ ]:
print(df_icarus[["review_id", "review_text", "voted_up", "playtime_hours"]].head())
print(df_icarus["voted_up"].value_counts(normalize=True))
print(df_icarus["review_length"].describe())

In [ ]:
import re

def is_mostly_english(text, threshold=0.8):
    text = str(text)
    if not text.strip():
        return False
    
    # 只统计字母和空格后的比例
    english_chars = re.findall(r"[A-Za-z\s]", text)
    ratio = len(english_chars) / max(len(text), 1)
    return ratio >= threshold

df_icarus["is_english"] = df_icarus["review_text"].apply(is_mostly_english)

print(df_icarus["is_english"].value_counts())

df_icarus_en = df_icarus[df_icarus["is_english"] == True].copy()
print(df_icarus_en.shape)

In [ ]:
print(df_icarus_en[["review_text"]].sample(10, random_state=42))

In [ ]:
df_icarus = df_icarus_en.copy()

In [ ]:
print(df_icarus.groupby("voted_up")["playtime_hours"].describe())
print(df_icarus.groupby("voted_up")["review_length"].describe())

In [ ]:
import matplotlib.pyplot as plt

q95 = df_icarus["playtime_hours"].quantile(0.95)
df_icarus_trim = df_icarus[df_icarus["playtime_hours"] <= q95]

df_icarus_trim.boxplot(column="playtime_hours", by="voted_up")
plt.title("Icarus: Playtime Hours by Recommendation")
plt.suptitle("")
plt.xlabel("Recommended")
plt.ylabel("Playtime Hours")
plt.show()

In [ ]:
import re
from collections import Counter

# 1. 文本清洗
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_icarus["clean_text"] = df_icarus["review_text"].apply(clean_text)

# 2. Icarus 最终版停用词
custom_stopwords_icarus = {
    # 基础英语虚词
    "the","a","an","and","or","but","if","then","so","of","to","in","on","for","with","at","by","from",
    "is","it","this","that","these","those","be","been","being","am","are","was","were",
    "i","me","my","mine","you","your","yours","he","she","they","them","their","we","our","us",
    "as","not","no","yes","do","does","did","have","has","had","can","could","would","should","shall",

    # 常见评价废词
    "good","great","best","bad","nice","okay","ok","love","like","liked",
    "really","very","pretty","quite","much","many","some","any","all","every","more","most","less",
    "still","even","just","also","well","too","only","there","here","what","when","where","why","how",

    # 口语残留
    "im","ive","id","youre","dont","didnt","cant","couldnt","wont","wasnt","isnt","arent","don","its",

    # 游戏评论高频废词
    "game","games","play","played","playing","player","players","hours","hour","time",
    "one","two","first","second","get","got","go","going","new","old","make","made","say","said",
    "will","would","way","thing","things","something","anything","everything","someone","anyone",

    # 你前面筛出来该删的
    "out","into","better","lot","other","which","want",
    "because","now","over","after","about","than","back",
    "keep","feel","feels","again",

    # 游戏名相关
    "icarus"
}

# 3. tokenize
def tokenize_and_filter(text):
    tokens = text.split()
    tokens = [w for w in tokens if w not in custom_stopwords_icarus and len(w) >= 3]
    return tokens

df_icarus["tokens"] = df_icarus["clean_text"].apply(tokenize_and_filter)

In [ ]:
def get_top_words_from_tokens(token_series, top_n=20):
    counter = Counter()
    for tokens in token_series.dropna():
        counter.update(tokens)
    return counter.most_common(top_n)

positive_words_icarus = get_top_words_from_tokens(
    df_icarus[df_icarus["voted_up"] == True]["tokens"], top_n=20
)

negative_words_icarus = get_top_words_from_tokens(
    df_icarus[df_icarus["voted_up"] == False]["tokens"], top_n=20
)

print("Top positive words (Icarus):")
print(positive_words_icarus)

print("\nTop negative words (Icarus):")
print(negative_words_icarus)

In [ ]:
focus_keywords_icarus = [
    "survival",
    "fun",
    "building",
    "world",
    "friends",
    "dlc",
    "content",
    "missions",
    "update",
    "issues"
]

In [ ]:
from collections import Counter

def get_keyword_counts(token_series, keyword_set):
    counter = Counter()
    for tokens in token_series.dropna():
        for t in tokens:
            if t in keyword_set:
                counter[t] += 1
    return counter

icarus_theme_keywords = {
    "survival", "fun", "world", "open", "building", "base", "crafting",
    "friends", "recommend", "worth", "dlc", "content", "buy", "update",
    "updates", "issues", "bugs", "devs", "missions", "mission", "map",
    "animals", "items",
    # 新加的
    "grind", "crash", "boring", "optimization", "lag", "fps", "tutorial"
}

positive_keyword_counts_icarus = get_keyword_counts(
    df_icarus[df_icarus["voted_up"] == True]["tokens"],
    icarus_theme_keywords
)

negative_keyword_counts_icarus = get_keyword_counts(
    df_icarus[df_icarus["voted_up"] == False]["tokens"],
    icarus_theme_keywords
)

print("Positive keyword counts (Icarus):")
print(positive_keyword_counts_icarus.most_common())

print("\nNegative keyword counts (Icarus):")
print(negative_keyword_counts_icarus.most_common())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

focus_keywords_icarus = [
    "survival",
    "fun",
    "building",
    "world",
    "friends",
    "dlc",
    "content",
    "missions",
    "update",
    "issues"
]

pos_dict_icarus = dict(positive_keyword_counts_icarus)
neg_dict_icarus = dict(negative_keyword_counts_icarus)

compare_df_icarus = pd.DataFrame({
    "keyword": focus_keywords_icarus,
    "positive": [pos_dict_icarus.get(k, 0) for k in focus_keywords_icarus],
    "negative": [neg_dict_icarus.get(k, 0) for k in focus_keywords_icarus]
})

print(compare_df_icarus)

compare_df_icarus = compare_df_icarus.sort_values("positive", ascending=True)

y = np.arange(len(compare_df_icarus))
height = 0.38

plt.figure(figsize=(10, 6))
plt.barh(y - height/2, compare_df_icarus["positive"], height=height, label="Positive Reviews")
plt.barh(y + height/2, compare_df_icarus["negative"], height=height, label="Negative Reviews")

plt.yticks(y, compare_df_icarus["keyword"])
plt.xlabel("Count")
plt.ylabel("Keyword")
plt.title("Icarus: Positive vs Negative Review Themes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
positive_focus_icarus = [
    "survival", "fun", "world", "friends", "open",
    "crafting", "worth", "solo", "amazing", "recommend"
]

In [ ]:
negative_focus_icarus = [
    "dlc", "content", "map", "missions", "devs",
    "issues", "updates", "update", "items",
    # 新加的
    "grind", "crash", "boring", "lag", "optimization"
]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

positive_focus_icarus = [
    "survival", "fun", "world", "friends", "open",
    "crafting", "recommend", "worth", "amazing", "solo"
]

pos_dict_icarus = dict(positive_keyword_counts_icarus)

pos_single_df_icarus = pd.DataFrame({
    "keyword": positive_focus_icarus,
    "count": [pos_dict_icarus.get(k, 0) for k in positive_focus_icarus]
})

# 去掉 0
pos_single_df_icarus = pos_single_df_icarus[pos_single_df_icarus["count"] > 0]
pos_single_df_icarus = pos_single_df_icarus.sort_values("count", ascending=True)

plt.figure(figsize=(9, 5.5))
plt.barh(pos_single_df_icarus["keyword"], pos_single_df_icarus["count"])
plt.xlabel("Count")
plt.ylabel("Keyword")
plt.title("Icarus: Positive Review Themes")
plt.tight_layout()
plt.show()

In [ ]:
negative_focus_icarus = [
    "dlc", "content", "map", "missions", "devs",
    "issues", "updates", "update", "items", "need"
]

neg_dict_icarus = dict(negative_keyword_counts_icarus)

neg_single_df_icarus = pd.DataFrame({
    "keyword": negative_focus_icarus,
    "count": [neg_dict_icarus.get(k, 0) for k in negative_focus_icarus]
})

# 去掉 0
neg_single_df_icarus = neg_single_df_icarus[neg_single_df_icarus["count"] > 0]
neg_single_df_icarus = neg_single_df_icarus.sort_values("count", ascending=True)

plt.figure(figsize=(9, 5.5))
plt.barh(neg_single_df_icarus["keyword"], neg_single_df_icarus["count"])
plt.xlabel("Count")
plt.ylabel("Keyword")
plt.title("Icarus: Negative Review Themes")
plt.tight_layout()
plt.show()

In [ ]:
dlc_positive = df_icarus[
    (df_icarus["voted_up"] == True) &
    (df_icarus["clean_text"].str.contains(r"\bdlc\b", na=False))
][["review_text"]]

dlc_negative = df_icarus[
    (df_icarus["voted_up"] == False) &
    (df_icarus["clean_text"].str.contains(r"\bdlc\b", na=False))
][["review_text"]]

print("Positive DLC mentions:")
print(dlc_positive.head(20))

print("\nNegative DLC mentions:")
print(dlc_negative.head(20))

In [ ]:
print(df_icarus.groupby("voted_up")["playtime_hours"].describe())
print("\nMedian playtime by review outcome:")
print(df_icarus.groupby("voted_up")["playtime_hours"].median())

In [ ]:
import numpy as np
import pandas as pd

# Unified lifecycle bands for comparison across Icarus, PoE, and Apex
bins = [0, 10, 50, 100, 300, 1000, np.inf]
labels = ["0-10h", "10-50h", "50-100h", "100-300h", "300-1000h", "1000h+"]

df_icarus["playtime_group"] = pd.cut(
    df_icarus["playtime_hours"],
    bins=bins,
    labels=labels,
    right=False
)

playtime_sort_map = {
    "0-10h": 1,
    "10-50h": 2,
    "50-100h": 3,
    "100-300h": 4,
    "300-1000h": 5,
    "1000h+": 6
}

df_icarus["playtime_sort"] = df_icarus["playtime_group"].astype(str).map(playtime_sort_map)


In [ ]:
thresholds = [50, 100, 200, 500]

for t in thresholds:
    share_all = (df_icarus["playtime_hours"] >= t).mean()
    share_by_sentiment = df_icarus.groupby("voted_up").apply(
        lambda x: (x["playtime_hours"] >= t).mean()
    )

    print(f"\n=== {t}+ hours ===")
    print(f"All reviews: {share_all:.2%}")
    print("By review outcome:")
    print(share_by_sentiment)

In [ ]:
icarus_bin_summary = (
    df_icarus.groupby("playtime_group", observed=True)
    .agg(
        review_count=("voted_up", "count"),
        positive_rate=("voted_up", "mean"),
        negative_rate=("voted_up", lambda x: 1 - x.mean()),
        median_hours=("playtime_hours", "median")
    )
    .reset_index()
    .sort_values("playtime_group", key=lambda s: s.astype(str).map(playtime_sort_map))
)

print(icarus_bin_summary)


In [ ]:
short_negative_icarus = df_icarus[
    (df_icarus["playtime_bin"] == "0–10h") &
    (df_icarus["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(short_negative_icarus.head(30))

In [ ]:
long_negative_icarus_200 = df_icarus[
    (df_icarus["playtime_hours"] >= 200) &
    (df_icarus["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(long_negative_icarus_200.head(30))

In [ ]:
long_negative_icarus_500 = df_icarus[
    (df_icarus["playtime_hours"] >= 500) &
    (df_icarus["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(long_negative_icarus_500.head(30))

In [ ]:
mid_long_negative_icarus = df_icarus[
    (df_icarus["playtime_bin"] == "200–500h") &
    (df_icarus["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(mid_long_negative_icarus.head(30))

In [ ]:
ultra_long_negative_icarus = df_icarus[
    (df_icarus["playtime_bin"] == "500h+") &
    (df_icarus["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(ultra_long_negative_icarus.head(30))

In [ ]:
import pandas as pd

# 导出正评关键词频率
pos_export = pd.DataFrame(
    positive_keyword_counts_icarus.most_common(),
    columns=["keyword", "positive_count"]
)

# 导出负评关键词频率
neg_export = pd.DataFrame(
    negative_keyword_counts_icarus.most_common(),
    columns=["keyword", "negative_count"]
)

# 合并成一个文件
keyword_export = pos_export.merge(neg_export, on="keyword", how="outer").fillna(0)
keyword_export["positive_count"] = keyword_export["positive_count"].astype(int)
keyword_export["negative_count"] = keyword_export["negative_count"].astype(int)

# 保存
keyword_export.to_csv("icarus_keyword_counts.csv", index=False, encoding="utf-8-sig")
print("已保存！")
print(keyword_export)

In [ ]:
# 重新导出关键词频率
pos_export = pd.DataFrame(
    positive_keyword_counts_icarus.most_common(),
    columns=["keyword", "positive_count"]
)
neg_export = pd.DataFrame(
    negative_keyword_counts_icarus.most_common(),
    columns=["keyword", "negative_count"]
)
keyword_export = pos_export.merge(neg_export, on="keyword", how="outer").fillna(0)
keyword_export["positive_count"] = keyword_export["positive_count"].astype(int)
keyword_export["negative_count"] = keyword_export["negative_count"].astype(int)
keyword_export.to_csv("icarus_keyword_counts.csv", index=False, encoding="utf-8-sig")
print("导出完成！")

In [ ]:
df_icarus.to_csv(
    "icarus_steam_reviews_english_filtered_1466.csv",
    index=False,
    encoding="utf-8-sig"
)
